# SEC-04 ARPスプーフィングと中間者攻撃

このノートブックでは、実際のネットワークへパケットを送らず、Pythonオブジェクトだけで構成した閉じたLAN上で次の順序を観察します。

1. 正常なARP解決とデータ配送
2. 偽装ARPリプライによるキャッシュ更新
3. 攻撃者を経由しながら通信が継続する状態
4. 競合検知と静的バインディングによる防御

> **安全上の注意**
> 本教材はシミュレータ専用です。許可のないネットワーク、端末、アカウントに対する試行へ転用しないでください。


## 1. 教材コードを準備する

Colabの一時環境へリポジトリを複製し、`security-course` サブプロジェクトを編集可能モードでインストールします。


In [ ]:
!test -d /content/network-simulator || git clone -q https://github.com/flyby-yunakayama/network-simulator.git /content/network-simulator
!test -d /content/network-simulator/security-course || (cd /content/network-simulator && git fetch -q origin agent/security-course-foundation && git checkout -q agent/security-course-foundation)
%cd /content/network-simulator/security-course
!python -m pip install -q -e .


## 2. 防御なし・防御ありを同じ条件で比較する

`compare_defense()` は、トポロジ、メッセージ、実行順序を固定したまま、防御ポリシーだけを切り替えます。これにより、結果の差を防御の効果として解釈できます。


In [ ]:
from pprint import pprint

from conecolab_security import compare_defense

unprotected, protected = compare_defense()

print("防御なし")
pprint(unprotected.summary(), sort_dicts=False)
print("\n防御あり")
pprint(protected.summary(), sort_dicts=False)


### 結果の読み方

- `attack_succeeded` は、攻撃後のメッセージを攻撃者が観測したかを表します。
- `service_available_after_attack` は、本来の宛先にもメッセージが到達したかを表します。
- `detection_alerts` は、同じIPアドレスに対する複数MAC主張から生成された、重複抑止後の競合アラート数です。
- `rejected_arp_updates` は、防御ポリシーが拒否したARPキャッシュ更新数です。

攻撃者がデータを中継する場合、サービスは動き続けるため、可用性だけを監視しても中間者状態を見逃し得る点に注目してください。


## 3. イベント列から原因を追う

攻撃開始後のイベントだけを抽出し、ARPキャッシュ更新、検知、データ中継の順序を確認します。


In [ ]:
start = next(
    event.sequence
    for event in unprotected.events
    if event.kind == "attack.started"
)

for event in unprotected.events:
    if event.sequence >= start:
        print(
            f"{event.sequence:02d}",
            f"{event.kind:<20}",
            f"{event.actor:<22}",
            event.message,
            dict(event.data),
        )


## 4. 演習

1. `detector.alert` が発生しても、防御なしでは攻撃が成功する理由をイベント列から説明してください。
2. `src/conecolab_security/arp_lab.py` の `StaticBindingGuard` を読み、保護対象に含まれないIPアドレスの更新を許可している理由とリスクを考えてください。
3. 攻撃者の `forwarding` を `False` にした場合、機密性と可用性の指標がどう変わるか予想してからコードで確認してください。
4. 「一定時間内に未要求のARPリプライがN回以上」という別の検知ルールを仕様として記述してください。実装前に、正常通信を誤検知しそうな条件も列挙します。

### 現実のネットワークとの差

このモデルは、OSごとのARPキャッシュ更新規則、タイムアウト、スイッチ機能、暗号化通信、無線LAN、複数セグメントなどを省略しています。実環境の挙動を断定するときは、対象OS・機器の公式資料とパケットキャプチャを別途確認してください。


## 5. 参考資料

- `[RFC-0826]` RFC Editor, *An Ethernet Address Resolution Protocol*
- `[RFC-5227]` RFC Editor, *IPv4 Address Conflict Detection*
- `[MITRE-T1557-002]` MITRE ATT&CK, *Adversary-in-the-Middle: ARP Cache Poisoning*
- `[MITRE-DET0387]` MITRE ATT&CK, Detection Strategy DET0387

URLと参照日は `references/sources.json` を正本とします。
